# 1. Context

This notebook does adhoc experimentation with Post OCR Corrections

# 2. Imports

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
from dotenv import load_dotenv

In [3]:
import os

In [4]:
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [5]:
# import torch
# torch.cuda.current_device()

In [6]:
load_dotenv()

True

# 3. Model Exploration

## 3.1. Gemma 3

### 3.1.1. Gemma-3 270M 

In [7]:
model_id = "google/gemma-3-1b-it"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="mps")

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)


In [ ]:
outputs = model.generate(**inputs, max_new_tokens=100)

In [ ]:
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [9]:
OCR_CORRECTOR_PROMPT = """You are a Post OCR Corrector Model. 
You will be provided with text to be corrected in {text} placeholder.
Rectify any error present and provide corrected output in {corrected_text} placeholder"""

In [10]:
text = "अतः आप ज्येष्ठ मास की शुक्ल पक्ष की निर्जुला नाम की एक ही एकादशी का व्रत करो और तुम्हें वर्ष की समस्त एकादशियों का फल प्राप्त होगा"

In [13]:
messages = [
    {"role": "user", "content": f"{OCR_CORRECTOR_PROMPT}\n\ntext: {text}"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=100)

In [14]:
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

अतः आप ज्येष्ठ मास की शुक्ल पक्ष की निर्जुला नाम की एक ही एकादशी का व्रत करो और तुम्हें वर्ष की समस्त एकादशियों का फल प्राप्त होगा।<end_of_turn>


In [ ]:
class postCorrectorLLM():
    """Class For OCR Post Correction Using a LLM"""

    def __init__(self, model_id: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda")
        self.model_id = model_id
        self.correction_prompt = None

        
    